# Bring your own decisions

CircLS puts every compilation decision behind a hook: you can inject
your own **mapping**, **orientation**, **lifetimes**, **routing**,
**schedule**, or **re-selection**, and the pipeline builds and
verifies the physical circuit around *your* choices.  Injected
decisions pass the same legality checks and the same construction
oracle as the built-in ones — a bad injection fails loudly, never
silently.

| decision | off switch | inject your own |
|---|---|---|
| re-selection | `measure_reduction=False` | `reselector=` |
| PPM ordering | `step_scheduling=False` | `scheduler=` |
| patch placement | `assignment="row_major"` | `placement=` |
| patch orientation | — | `orientation=` |
| corridor routing | — | `router=` (or `PPMStep.route`) |
| patch lifetimes | `liveness=`, `keep_patches=` | `lifetime=` |

Contracts and validation rules: `docs/API_HOOKS.md`.


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from circls.pipeline import compile_qasm

QASM = '''
OPENQASM 2.0;
include "qelib1.inc";
qreg q[3];
creg c[3];
h q[0];
cx q[0], q[1];
cx q[1], q[2];
measure q[0] -> c[0];
measure q[1] -> c[1];
measure q[2] -> c[2];
'''


## Read the compiler's decisions off a baseline compile

Every decision is visible on the result, so the natural workflow is:
compile once, read a decision out, change it, feed it back in.

(The demos compile with `measure_reduction=False`: on this GHZ
program, re-selection collapses every joint PPM into a
single-patch readout — correct, but it leaves no routing or
lifetime decisions to look at.)


In [ ]:
base = compile_qasm(QASM, distance=3, measure_reduction=False)

print('placement :', base.placement)
print('lifetimes :', base.lifetimes)
print('routes    :', base.routes)
print('volume    :', base.stats()['V1_volume_blocks'], 'blocks')


## Your own mapping

`placement=` takes `{patch name: coarse cell}` and must cover every
patch, one cell each.  Here we hand the compiler the baseline's map
with two patches swapped; any dict of legal cells works the same way.


In [ ]:
mine = dict(base.placement)
a, b = sorted(mine)[0], sorted(mine)[-1]
mine[a], mine[b] = mine[b], mine[a]

out = compile_qasm(QASM, distance=3, measure_reduction=False, placement=mine)
print('placement :', out.placement)
print('volume    :', out.stats()['V1_volume_blocks'], 'blocks')


## Your own orientation

`orientation=` fixes a patch's birth orientation
(`X_horizontal` / `X_vertical`).  A partial dict is fine — unnamed
patches keep the derived orientation.  (|Y> gadget patches cannot be
overridden: their birth layout is protocol-fixed.)


In [ ]:
nm = sorted(base.placement)[0]
cur = next(s.orientation for s in base.experiment.patches if s.name == nm)
flip = 'X_vertical' if cur == 'X_horizontal' else 'X_horizontal'

out = compile_qasm(QASM, distance=3, measure_reduction=False, orientation={nm: flip})
print(nm, ':', cur, '->', next(s.orientation for s in out.experiment.patches
                               if s.name == nm))


## Your own lifetimes

`lifetime=` takes `{patch name: (init_layer, free_layer)}` in PPM-layer
indices.  The pair is a schedule: the patch is allocated at
`init_layer` (idling until its first use) and, with `liveness=` on,
measured out at `free_layer`.  Overrides may only widen — the patch
must exist at every layer that uses it.


In [ ]:
nm, (fu, lu) = sorted(base.lifetimes.items())[-1]
out = compile_qasm(QASM, distance=3, measure_reduction=False, lifetime={nm: (0, lu)})
print(f'{nm}: derived window ({fu},{lu}) -> allocated up front (0,{lu})')
print('lifetimes :', out.lifetimes)


## Your own routing

`router=` is a strategy hook: it is consulted once per step that needs
a corridor, with the live patch specs.  Return a list of coarse cells
to force that corridor (it still runs through the full construction
oracle), or `None` to fall back to the internal search.


In [ ]:
calls = []

def declining(live_specs, step):
    calls.append(step.interaction_type)
    return None            # fall back to the internal search

out = compile_qasm(QASM, distance=3, measure_reduction=False, router=declining)
print('router consulted for:', calls)


In [ ]:
# replay the corridors a previous compile verified
verified = {tuple(sorted(s.interaction_type)): cells
            for s, cells in zip(base.experiment.ppm_sequence, base.routes)
            if cells}

def replay(live_specs, step):
    return verified.get(tuple(sorted(step.interaction_type)))

out = compile_qasm(QASM, distance=3, measure_reduction=False, router=replay)
print('routes    :', out.routes)


## Your own schedule

`scheduler=` maps the front-end program to `(program, perm)` where
`perm[new_pos] = old_pos`.  The contract is checked: only commuting
operations may swap, anticommuting pairs keep their order, weight-1
fixed operations keep their absolute positions.


In [ ]:
def keep_program_order(pre):
    return pre, list(range(len(pre.ops)))

out = compile_qasm(QASM, distance=3, measure_reduction=False, scheduler=keep_program_order)
print('volume    :', out.stats()['V1_volume_blocks'], 'blocks')


## Enter with your own PPM sequence

You do not have to start from QASM.  `compile_ppm_sequence` accepts a
`PPMProgram` — the output of any PBC front-end — and skips QASM
loading, re-selection and scheduling; the `placement=`,
`orientation=`, `lifetime=` and `router=` hooks apply unchanged.
(One default difference: this entry leaves `parallel_steps` at
`False`; pass `parallel_steps=True` for shared execution windows.)


In [ ]:
from circls.interop.ir.gosc_gadgets import OutBit, PPMProgram, ProgramOp
from circls.pipeline import compile_ppm_sequence

prog = PPMProgram(
    num_data=3,
    ops=[
        # jointly measure X0 X1, then X1 X2, then read every qubit out
        ProgramOp('mpp', {0: 'X', 1: 'X'}),
        ProgramOp('mpp', {1: 'X', 2: 'X'}),
        ProgramOp('mpp', {0: 'Z'}),
        ProgramOp('mpp', {1: 'Z'}),
        ProgramOp('mpp', {2: 'Z'}),
    ],
    gadgets=[],
    # program bit i = record[rec] XOR flip: here bit i is just the
    # i-th measurement record, unflipped
    out_bits=[OutBit(rec=k, flip=0) for k in range(5)],
)
out = compile_ppm_sequence(prog, distance=3)
print('placement :', out.placement)
print('routes    :', out.routes)


## What happens on a bad injection

Validation is loud: unknown patch names, duplicate cells, illegal
orientation values, narrowed lifetimes, and illegal corridors all
raise before or during construction — never a silently wrong circuit.


In [ ]:
try:
    compile_qasm(QASM, distance=3, measure_reduction=False,
                 placement={nm: (0, 0) for nm in base.placement})
except ValueError as e:
    print('rejected:', e)
